# 01 · Define & Explore — the conserved neutralizing epitope + VHH metrics

**Standard slot:** *define & explore.* **For Project 14 this means:** clean the viral antigen, pick a
**conserved neutralizing epitope** + a humanized VHH **framework**, fix the metrics, and run a tiny
mock VHH batch as your hello-world (D0).

> **Defensive framing.** Every design is steered to a conserved epitope to *block* the virus. Enhancing
> viral fitness/affinity/escape is out of scope (`README.md` → Responsible research, `MASTER_BLUEPRINT.md §7`).

Run `00_setup.ipynb` first in this session.

## The metrics, precisely (nanobody design)

| Metric | Means | Does **not** mean |
|--------|-------|-------------------|
| `pae_interaction` (AF2-Multimer) | interface confidence (antibody cutoff ≤ 12; lower better) | measured affinity |
| interface pLDDT | local confidence | stability / K_D |
| scRMSD | designed-vs-predicted VHH backbone self-consistency | binding |
| CDR geometry RMSD | loop/Ramachandran sanity vs IgFold | function |
| developability (TAP/CamSol/humanness) | aggregation/solubility/immunogenicity **proxies** | a verdict (use real tools) |
| **worst-case breadth pae** | does the VHH hold across ALL strains? | best-case is not breadth |


## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Epitope + framework choice (EXAMPLE — verify on the real structure)

Choose hotspots on the **conserved neutralizing face** (e.g. the HA stem). The residues below are
**EXAMPLE placeholders** — derive the real ones from 4FQI + a cross-strain conservation analysis.

In [ ]:
import antibody_tools as ab

ANTIGEN = "HA_STEM"
EPITOPE = ab.parse_epitope("A18,A41,A45,A49")   # EXAMPLE conserved HA-stem residues — VERIFY from 4FQI
FRAMEWORK = ab.DEFAULT_FRAMEWORK                  # humanized VHH placeholder — verify/replace
print("antigen:", ANTIGEN, "| EXAMPLE epitope (verify):", EPITOPE)
print("framework:", FRAMEWORK["name"])

## Hello-world: a tiny mock VHH batch

Develop the plumbing with the deterministic `mock` backend (no GPU). Switch `tool="rfantibody"` on an
A100 (see `MANUAL.md §2`). **Mock numbers/sequences are SYNTHETIC — never report them.**

In [ ]:
vhhs = ab.design_vhh_cdrs(ANTIGEN, EPITOPE, framework=FRAMEWORK, n=5, tool="mock")
ab.score_designs(vhhs, tool="mock")
d = vhhs[0]
print("example:", d.design_id, "| CDR3=", d.cdr3, "(", len(d.cdr3), "aa )")
print("  af2: pae_interaction=", d.pae_interaction, "scrmsd=", d.scrmsd, "cdr_geom=", d.cdr_geom)
print("  dev: tap=", d.tap_score, "camsol_like=", d.camsol_like, "humanness=", d.humanness, "(TEACHING HEURISTICS)")
print("  epitope overlap (neutralization proxy):", ab.epitope_overlap(d.contact_residues, EPITOPE))
print("[reminder] every number above is SYNTHETIC (mock).")

## Breadth concept: epitope conservation across strains `[core]`

A neutralizing VHH is broadly protective only if its epitope is **conserved** across strains. The
snippet below is **EXAMPLE_DATA** (toy aligned fragments) to demonstrate `epitope_conservation()`;
in your project you align a real strain panel and score the columns under your epitope.

In [ ]:
# EXAMPLE_DATA — toy aligned antigen fragments (NOT real sequences), positions 1..10.
EXAMPLE_STRAINS = {
    "H1": "GLFGAIAGFI",
    "H3": "GLFGAIAGFI",
    "H5": "GLFGAIAGFL",   # a change at position 10
}
stem_epitope = [2, 5, 8]   # toy conserved-stem positions
head_epitope = [10]        # toy variable-head position
print("conserved (stem) epitope conservation:", ab.epitope_conservation(stem_epitope, EXAMPLE_STRAINS))
print("variable (head) epitope conservation: ", ab.epitope_conservation(head_epitope, EXAMPLE_STRAINS))
print("[EXAMPLE_DATA] toy demo of the breadth rationale — replace with the real strain panel.")

## D0 checklist
- [ ] Conserved-epitope map + justification of the chosen neutralizing face; humanized framework chosen.
- [ ] One-paragraph definition of each metric **with** its 'does not mean' note.
- [ ] One reproduced mock mini-run (VHHs scored, developability + breadth rationale shown).
- [ ] `LOG.md` entry: tool version, GPU, seed.

**Next:** `02_generate.ipynb` — the RFantibody CDR design campaign.